In [41]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder,LabelEncoder
from sklearn.model_selection import StratifiedKFold,cross_val_predict
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

import joblib

In [42]:
df = pd.read_csv('/content/Cyclone.csv',skiprows=[1], na_values=[' ', ''])

/tmp/ipykernel_2247/3480873925.py:1: DtypeWarning: Columns (12,17,18,21,22,61,66,77,139) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/Cyclone.csv',skiprows=[1], na_values=[' ', ''])


In [43]:
df.shape

(62605, 174)

In [44]:
df.head().T

,0,1,2,3,4
SID,1842298N11080,1842298N11080,1842298N11080,1842298N11080,1842298N11080
SEASON,1842,1842,1842,1842,1842
NUMBER,1,1,1,1,1
BASIN,NI,NI,NI,NI,NI
SUBBASIN,BB,BB,BB,BB,BB
...,...,...,...,...,...
USA_SEARAD_SE,NaN,NaN,NaN,NaN,NaN
USA_SEARAD_SW,NaN,NaN,NaN,NaN,NaN
USA_SEARAD_NW,NaN,NaN,NaN,NaN,NaN
STORM_SPEED,9.0,9.0,9.0,9.0,9.0


In [45]:
df.describe()

,SEASON,NUMBER,LAT,LON,WMO_WIND,WMO_PRES,DIST2LAND,LANDFALL,USA_LAT,USA_LON,...,BOM_GUST_PER,REUNION_GUST,REUNION_GUST_PER,USA_SEAHGT,USA_SEARAD_NE,USA_SEARAD_SE,USA_SEARAD_SW,USA_SEARAD_NW,STORM_SPEED,STORM_DIR
count,62605.000000,62605.000000,62605.000000,62605.000000,7660.000000,7958.000000,62605.000000,60758.000000,30646.000000,30646.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,62563.000000,62563.000000
mean,1952.935596,57.101030,17.622706,82.998516,41.933159,991.600779,216.912499,210.387817,16.178705,82.264296,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.159583,231.124147
std,40.916771,31.309144,6.180089,17.828134,22.260115,14.954082,270.053131,266.740292,6.859999,23.242356,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.575763,117.140942
min,1842.000000,1.000000,0.700000,-87.700000,3.000000,890.000000,0.000000,0.000000,0.700000,-87.700000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
25%,1922.000000,31.000000,13.000000,77.900000,25.000000,988.000000,0.000000,0.000000,11.500000,72.800000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,185.000000
50%,1957.000000,51.000000,18.300000,84.700000,35.000000,996.000000,112.000000,103.000000,15.600000,84.800000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.000000,285.000000
75%,1986.000000,84.000000,21.900000,88.600000,50.000000,1000.000000,361.000000,350.000000,20.400000,89.600000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000,305.000000
max,2025.000000,140.000000,83.000000,163.700000,140.000000,1012.000000,2271.000000,2269.000000,83.000000,163.700000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.000000,360.000000


In [46]:
df['SUBBASIN'].value_counts()

,count
SUBBASIN,
BB,42320
AS,15276
MM,4527
GM,25
CS,8


In [47]:
df['USA_WIND'] = pd.to_numeric(df['USA_WIND'], errors='coerce')
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df['initial_wind_proxy'] = df['WMO_WIND'].fillna(df['USA_WIND'])

In [66]:
storms = df.groupby('SID').agg(
    season=('SEASON', 'first'),
    max_wind=('WMO_WIND', 'max'),
    subbasin=('SUBBASIN', 'first'),
    start_month=('ISO_TIME', 'first'),
    genesis_lat=('LAT', 'first'),
    genesis_lon=('LON', 'first'),
    initial_wind=('initial_wind_proxy', 'first'),
).reset_index()

In [67]:
storms['month'] = pd.to_datetime(
    storms['start_month'], errors='coerce'
).dt.month

In [68]:
storms.shape

(1847, 9)

In [69]:
storms['max_wind'].describe()

,max_wind
count,426.000000
mean,45.955399
std,27.294910
min,4.000000
25%,25.000000
50%,30.000000
75%,55.000000
max,140.000000


In [70]:
def wind_to_severity(w):
    if pd.isna(w): return None
    w = float(w)
    if w >= 90:   return 'High'
    elif w >= 48: return 'Medium'
    else:         return 'Low'

In [71]:
storms['severity'] = storms['max_wind'].apply(wind_to_severity)
storms = storms[storms['severity'].notna()]

In [72]:
storms['severity'].value_counts()

,count
severity,
Low,294
Medium,83
High,49


In [73]:
def get_season(m):
    if pd.isna(m): return 'Unknown'
    m = int(m)
    if m in [6,7,8,9]: return 'Monsoon'
    elif m in [3,4,5]: return 'PreMonsoon'
    elif m in [10,11]: return 'PostMonsoon'
    else: return 'Winter'

In [74]:
storms['season'] = storms['month'].apply(get_season)

In [75]:
features = ['season', 'subbasin', 'month', 'genesis_lat', 'genesis_lon', 'initial_wind']

In [76]:
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value = -1
)

In [77]:
storms[['season','subbasin']] = encoder.fit_transform(
    storms[['season','subbasin']].astype(str)
)

In [78]:
le = LabelEncoder()
y = le.fit_transform(storms['severity'])
X = storms[features]

In [79]:
model = XGBClassifier(
    n_estimators=200, max_depth=4,
    missing=float('nan'),
    eval_metric='mlogloss', random_state=42, verbosity=0
)

In [80]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
preds = cross_val_predict(model, X, y, cv=cv)
print(classification_report(y, preds, target_names=le.classes_))

              precision    recall  f1-score   support

        High       0.18      0.16      0.17        49
         Low       0.76      0.81      0.78       294
      Medium       0.21      0.17      0.19        83

    accuracy                           0.61       426
   macro avg       0.38      0.38      0.38       426
weighted avg       0.58      0.61      0.60       426



In [81]:
model.fit(X, y)
joblib.dump(model, 'model3_cyclone_severity.pkl')
joblib.dump(encoder,   'model3_encoder.pkl')
joblib.dump(le,    'model3_label_encoder.pkl')
print("Model 3 saved.")

Model 3 saved.
